<img src="../logo.png" alt="Vegeta" width="240">

# ALBATROS — a maritime turbojet transport drone (50)

A vessel a few hundred kilometres out needs a part, a pump, medicine — now. **ALBATROS** is a fast jet drone that
carries it there: it leaves a ship's catapult or a runway ashore, cruises high on two small turbojets, slows over the
vessel and releases a **cargo multirotor out of its belly**, which flies the cargo down to the deck and stays there.
ALBATROS flies home and lands ashore.

What this notebook settles:

1. **The bay**: one cargo hexacopter, or up to three small quadcopters?
2. **The engines**: notebook 28's turbojet cycle (`vegeta.boreas.microjet`), with larger classes added to its catalogue.
3. **How it gets airborne**: a catapult or runway launch (the baseline), against a vertical take-off on jets with
   vectored thrust, and a jet tail-sitter. Each layout is closed on its own take-off mass for the same mission.
4. **The fuel**: by phase, and how it moves with the cargo, the distance, the altitude and the wind.
5. **The wing against gusts**: wing loading, sweep, flaperons that cancel gusts, and thrust vectoring.
6. **The launch and landing**, and **the release**: the folded hexacopter out of the bay, under a drogue, arms out,
   rotors running — and the height the pass must be flown at.
7. **The airframe in CAD**, and what happens if it ends up in the water.
8. **Medicine to a flooded town**: the bay loaded with six ÉMERILLONs (notebook 37's speed quadcopter) carrying
   vaccines and medicines, released in one pass over a flooded town and flown to the people on the roofs — in MuJoCo.

A fourth layout — electric lift rotors on booms for the deck, jets for the cruise — was sized in the first pass and
dropped: no propellers on Albatros. It stays in `albatros_flight.SET_ASIDE` and appears once, in section 4, as a number.

Everything here is analysis by first-order models (their limits are in each module's text). No CFD, no FEA.

In [ ]:
# DEBUG — ON BY DEFAULT: the expensive solvers are mocked (vegeta.mock). This notebook runs none of them: it is the
# same with DEBUG on or off.
DEBUG = True
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)        # the lab's folders on sys.path, the mocks

In [ ]:
import json, math, shutil
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vegeta import dedalus
from vegeta.boreas import microjet as mj
from vegeta.dedalus import viz as dviz
import albatros as al
import albatros_bay as B
import albatros_drop as DR
import albatros_flood as FLD
import albatros_flight as F
import albatros_gust as GU

ROOT = Path("../_runs/albatros"); shutil.rmtree(ROOT, ignore_errors=True); ROOT.mkdir(parents=True)
DATA = Path("../designs/data")
G = 9.80665
pd.set_option("display.max_colwidth", 80)

In [ ]:
import yanus                                       # Ozaru's requirement checks: what this notebook needs on this machine
from yanus import Package
NEEDS = [Package("vegeta.boreas"), Package("vegeta.dedalus")]
print(yanus.table(yanus.check_needs(NEEDS)) if NEEDS else "needs: nothing beyond the Python environment")

## 1. The mission — edit here

10 kg of cargo, 300 km out, and back on the same fuel with 15 % of it left. The cruise is at 6000 m: a turbojet's
range grows with the true airspeed it can hold for a given drag, and thin air gives that (section 5 shows what a
lower cruise costs). The pass over the vessel and the approach at home are a minute each. (`wing_loading` and
`cl_max` below are the vertical layouts'; the catapult layout carries its own, 120 kg/m² and 1.7.)

In [ ]:
REQ = F.Requirements(cargo_kg=10.0, radius_km=300.0, headwind_out=0.0, cruise_altitude_m=6000.0, reserve=0.15,
                     release_hover_s=60.0, landing_hover_s=60.0, aspect_ratio=10.0, sweep_deg=20.0)
BAY = "hexacopter"                      # the bay's load: "hexacopter" | "three quadcopters" | "two hexacopters"
FULL = True                             # the sensitivity tables of section 5 (about two minutes); False skips them
display(pd.Series({k: getattr(REQ, k) for k in REQ.__dataclass_fields__}, name="requirement").to_frame())

## 2. The bay: one hexacopter, or three quadcopters?

The multirotor's flight is short — out of the bay, down to the deck, a go-around in hand — so it is sized by
lifting, not by endurance (`albatros_bay.size_multirotor`: momentum-theory hover power, a battery for five minutes,
thrust twice the weight). Folded, its arms lie fore and aft and the cargo box hangs under the hub.

The candidates for 10 kg: **one hexacopter** with all of it; **three quadcopters** with a third each; two half-size
hexacopters; and **three camera-class folding quadcopters as bought** (the small "DJI-like" kind), whose cargo is
their published take-off margin.

In [ ]:
OPTIONS = B.options(REQ.cargo_kg)
display(B.table(REQ.cargo_kg))
OPTION = OPTIONS[BAY]
HEX = OPTION.vehicles[0]
display(pd.Series({k: round(v, 2) for k, v in HEX.budget.items()}, name=f"{HEX.name}: mass [kg]").to_frame())

**Reading the table.**

- Three camera-class quadcopters carry about half a kilogram between them: they can bring a radio, a medicine pack
  or a line — not the cargo. They fit, with room to spare, and could ride along as scouts; they do not replace the
  cargo vehicle.
- Three *cargo* quadcopters carry the 10 kg, but each pays for its own flight controller, link and hook, so the
  three weigh more than the one hexacopter, and nothing heavier than 3.3 kg can go. A quadcopter that loses a rotor
  falls; the hexacopter stops the opposite rotor and flies on. Three vehicles almost never lose *everything*, but
  they deliver *all* of it less often (the probabilities are assumptions: `albatros_bay.delivery`).
- The bay of three is slimmer but 70 % longer.

**The bay is sized for one cargo hexacopter.** The same bay takes two half-size hexacopters or the three
quadcopters with a longer door; the choice can change per sortie (`BAY` above re-sizes the aircraft for it).

## 3. The engines: notebook 28's cycle

`vegeta.boreas.microjet` is one shaft, a radial compressor, an annular combustor, an axial turbine and a convergent
nozzle, calibrated to a datasheet point. Its catalogue stopped at the 200 N class; four larger classes are added
(300–800 N: the same layout at the same tip speed, a little less fuel per newton, slower to spool). Between the
classes the sizing scales an engine in air flow (`albatros_flight.scaled`): thrust, fuel and mass with the flow,
diameters with its square root.

The table shows why the mission is hard on a turbojet: at 90 m/s it turns about **3 % of the fuel's energy into
thrust work**. Speed and altitude are its friends; hovering on it is not.

In [ ]:
rows = {}
for name in F.CLASSES:
    e = mj.from_catalogue(name)
    s, c = mj.solve(e, e.rpm_max), mj.solve(e, 0.9 * e.rpm_max, 90.0, mj.isa(REQ.cruise_altitude_m))
    rows[name] = {"static thrust [N]": round(s.thrust), "fuel at full thrust [g/min]": round(s.fuel_flow_g_min),
                  "static TSFC [kg/(N h)]": round(s.tsfc, 3), "pressure ratio": round(s.pressure_ratio, 2),
                  "mass with systems [kg]": e.mass_kg + e.system_mass_kg, "Ø [mm]": round(e.outer_diameter * 1000),
                  f"thrust at 90 m/s, {REQ.cruise_altitude_m:.0f} m, 90 % rpm [N]": round(c.thrust),
                  "TSFC there [kg/(N h)]": round(c.tsfc, 3), "overall efficiency there [%]": round(100 * c.overall_efficiency(), 1)}
display(pd.DataFrame(rows).T)

## 4. Three ways into the air, sized for the same mission

`albatros_flight.size` finds, for each layout, the smallest take-off mass at which the aircraft built for that mass
— engines scaled to it, wing from the wing loading, fuselage around the bay and the tanks — and fuelled for the
mission weighs no more than that mass.

- **catapult** — the baseline. A wing-borne jet: catapult from the ship (or a runway ashore), landing ashore. Two
  turbojets with two-axis vectoring nozzles. Over the vessel it slows to 1.3 × its stall speed for the release.
  Its wing must fly off the catapult: 120 kg/m² with the flaperons drooped.
- **vectored** — vertical on jets: the two cruise engines swivel their nozzles down, two larger lift jets in the
  fuselage carry the rest (shut down in the cruise, restarted over the vessel). It hovers for the release.
  No stall speed to meet: its wing is loaded 250 kg/m².
- **tail-sitter** — four engines at the tail with vectoring nozzles; it stands on its fins.

In [ ]:
DESIGNS = F.compare(REQ, OPTION)
TABLE = F.table(DESIGNS)
display(TABLE)
budgets = pd.DataFrame({d.concept.name: pd.Series(d.budget) for d in DESIGNS.values()}).round(1)
display(budgets.astype(object).where(budgets.notna(), "-"))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
groups = {"structure and systems": ("structure", "actuators", "systems", "flotation", "fuel system", "launch", "tail-sitter", "lift-jet bay", "bay doors"),
          "engines and nozzles": ("engines", "lift jets", "nozzle"), "the bay's load": ("bay load",), "fuel": ("fuel",)}
names = [d.concept.name for d in DESIGNS.values()]
bottom = np.zeros(len(names))
for g, keys in groups.items():
    v = np.array([sum(m for k, m in d.budget.items() if any(k.startswith(q) or q in k for q in keys) and not (g != "fuel" and k == "fuel")
                      and not (g == "fuel" and k != "fuel")) for d in DESIGNS.values()])
    ax[0].bar(range(len(names)), v, bottom=bottom, label=g); bottom += v
ax[0].set_xticks(range(len(names))); ax[0].set_xticklabels([n.replace(", ", ",\n") for n in names], fontsize=8)
ax[0].set_ylabel("take-off mass [kg]"); ax[0].legend(); ax[0].set_title("What each layout weighs")
kinds = {"climb": ("climb",), "cruise": ("cruise",), "thrust-borne (hover, transition, restart)": ("take-off", "hover", "transition", "restart", "deck"),
         "start, launch, descent, pass, approach": ("start", "launch", "descent", "slow pass", "approach")}
bottom = np.zeros(len(names))
for g, keys in kinds.items():
    v = np.array([sum(p["fuel"] for k, p in d.flight.phases.items() if any(q in k for q in keys)) for d in DESIGNS.values()])
    ax[1].bar(range(len(names)), v, bottom=bottom, label=g); bottom += v
ax[1].bar(range(len(names)), [d.flight.fuel_left_kg for d in DESIGNS.values()], bottom=bottom, label="reserve", color="0.8")
ax[1].set_xticks(range(len(names))); ax[1].set_xticklabels([n.replace(", ", ",\n") for n in names], fontsize=8)
ax[1].set_ylabel("fuel [kg]"); ax[1].legend(fontsize=8); ax[1].set_title("Where the fuel goes")
plt.tight_layout(); plt.show()

**A vertical take-off on jets about doubles the aircraft and its fuel**; the tail-sitter multiplies it several times (its four large engines are far from their best in the cruise, and its number is the least steady of the three). The
catapult aircraft carries nothing for a hover: no lift jets, no 1.25 of thrust-to-weight, and a cleaner cruise on
small engines working near their best.

The numbers that pick a layout beside the fuel — gusts, the launch, the deck, the water — are next
(`albatros_gust.scorecard`), and `albatros_flight.choose` applies the rule: *the least fuel among the layouts that
pass their gates* (a catapult stroke of at most 12 m at 8 g for a wing-borne layout; for a thrust-borne one, hover
drift within 1 m in the side gust, a hover wind limit of 20 m/s, 0.25 m of height error over a heaving deck, and
standing on a rolling deck in 20 m/s).

In [ ]:
SCORE = GU.scorecard(DESIGNS, altitude_m=REQ.cruise_altitude_m)
display(SCORE)
CHOICE = F.choose(DESIGNS)
for k in CHOICE["by_fuel"]:
    d = DESIGNS[k]
    print(f"{d.concept.name:42s} fuel {CHOICE['fuel_kg'][k]:6.1f} kg   gates failed: {CHOICE['failed'][k] or 'none'}")
    for n in CHOICE["notes"][k]:
        print(f"{'':42s} note: {n}")
KEY = CHOICE["choice"]
D = DESIGNS[KEY]
FL = D.flight
print(f"\nALBATROS is the '{D.concept.name}' layout: {D.mtow_kg:.0f} kg at take-off, {FL.fuel_kg:.0f} kg ({FL.fuel_kg / 0.8:.0f} L) of kerosene.")

In [ ]:
# the layout that was set aside (electric lift rotors + jet cruise), once, for the record
ROTORS = F.size("lift_cruise", REQ, OPTION)
print(f"electric lift + jet cruise (set aside): {ROTORS.mtow_kg:.0f} kg at take-off, {ROTORS.flight.fuel_kg:.0f} kg of fuel, "
      f"a {ROTORS.battery_kg:.0f} kg lift battery, four {ROTORS.rotor_diameter_m:.2f} m rotors, {ROTORS.hover_power_w / 1000:.0f} kW in the hover")

## 5. The fuel

The sortie of the chosen layout, phase by phase. The climb to the cruise altitude is a quarter of the fuel; the
long glide down on idling engines gives part of it back as distance.

In [ ]:
display(FL.table())
tr = FL.track
fig, ax = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
tm = tr["t"] / 60
ax[0].plot(tm, tr["x"] / 1000, "C0"); ax[0].set_ylabel("distance from base [km]")
ax[1].plot(tm, tr["h"], "k"); ax[1].set_ylabel("height [m]")
ax[2].plot(tm, tr["mass"], "k"); ax[2].set_ylabel("mass [kg]")
ax[3].plot(tm, tr["fuel_flow"] * 60e3, "C1"); ax[3].set_ylabel("fuel flow [g/min]"); ax[3].set_xlabel("time [min]")
ax[2].annotate("the multirotor leaves", (FL.time_to_vessel_s / 60, D.mtow_kg - 0.5 * FL.fuel_kg), fontsize=9)
plt.tight_layout(); plt.show()
print(f"to the vessel in {FL.time_to_vessel_s / 60:.0f} min at {FL.cruise_speed_out:.0f} m/s ({FL.cruise_speed_out * 3.6:.0f} km/h); "
      f"sortie {FL.total_time_s / 60:.0f} min; fuel {FL.fuel_kg:.1f} kg = {FL.fuel_kg / 0.8:.0f} L, of it {FL.fuel_left_kg:.1f} kg reserve; "
      f"{(FL.fuel_kg - FL.fuel_left_kg) / (2 * REQ.radius_km) * 1000:.0f} g per km flown; top speed at altitude {F.top_speed(D, REQ):.0f} m/s")

**What moves the fuel.** Each row below is the chosen layout sized again with one thing changed. The sizing has
its own noise of about ±4 % of the take-off mass (discrete steps, the choice of running engines), so small
differences between rows are not findings.

In [ ]:
SENS = None
if FULL:
    def row(label, req, option=None):
        d = F.size(KEY, req, option or B.options(req.cargo_kg)[BAY])
        f = d.flight
        return {"case": label, "take-off [kg]": round(d.mtow_kg), "fuel [kg]": round(f.fuel_kg, 1), "fuel [L]": round(f.fuel_kg / 0.8),
                "engines": F.summary(d)["engines"], "cruise out [m/s]": round(f.cruise_speed_out), "to the vessel [min]": round(f.time_to_vessel_s / 60),
                "closes": bool(d.feasible)}
    rows = [row(f"{c:g} kg cargo, {r} km", replace(REQ, cargo_kg=c, radius_km=r)) for c in (2.0, 5.0, 10.0) for r in (150.0, 300.0)]
    rows += [row("three cargo quadcopters in the bay", REQ, OPTIONS["three quadcopters"]),
             row("cruise at 3000 m", replace(REQ, cruise_altitude_m=3000.0)), row("cruise at 8000 m", replace(REQ, cruise_altitude_m=8000.0)),
             row("15 m/s headwind out", replace(REQ, headwind_out=15.0)), row("cruise at 120 m/s or more", replace(REQ, min_cruise_speed=120.0)),
             row("300 km in all (150 km out and back, or one way to a far shore)", replace(REQ, radius_km=150.0))]
    SENS = pd.DataFrame(rows).set_index("case")
    display(SENS)

## 6. The wing against gusts

A sharp vertical gust `U` changes the lift by `ρ V U a S / 2` — the same force on any aircraft of that wing. So the
load factor it gives falls with the **wing loading** (Pratt's formula, `albatros_gust.gust_load`), and again with a
lower lift slope `a`: **sweep** and a moderate aspect ratio. The albatross does the same: for a bird, a heavily
loaded, slender wing.

The catapult sets how far this can go: the wing must fly at the catapult's end speed. The first plot shows the
trade — load factor step in a 10 m/s gust at the cruise, against the catapult stroke that wing loading needs.

What the wing's shape leaves, **the flaperons take out**: full-span, driven by the angle of attack measured at a
nose probe (which meets the gust before the wing does), they cancel the gust's lift as direct lift control
(`gust_alleviation`: a 1-cos gust flown in heave with a 40 ms actuator).

**Thrust vectoring** is for the slow end. The tail's moment falls with the dynamic pressure; the nozzles' does not
(third plot). With engines this small it does not replace the tail in flight: the two are equal only below the
stall speed. What it buys is the margin where the tail is weakest — the second after the catapult lets go, the
slow pass through the ship's wake, a go-around — and control of a wing that has stalled in a gust. The print
below gives its share at those speeds. (On the vertical layouts of section 4 it is the whole of the control.)

In [ ]:
P = D.params
chord = P["root_chord"] * (1 + P["taper"]) / 2000
rho_h = F.density(REQ.cruise_altitude_m)
V_C = FL.cruise_speed_out
ws = np.linspace(40, 300, 27)
dn = [GU.gust_load(w, V_C, chord=chord, aspect_ratio=REQ.aspect_ratio, sweep_deg=REQ.sweep_deg, U=10.0, rho=rho_h)["dn"] for w in ws]
dn0 = [GU.gust_load(w, V_C, chord=chord, aspect_ratio=REQ.aspect_ratio, sweep_deg=0.0, U=10.0, rho=rho_h)["dn"] for w in ws]
cl = D.airframe.cl_max
stroke = [(1.2 * math.sqrt(2 * w * G / (1.225 * cl))) ** 2 / (2 * 8 * G) for w in ws]
WS = D.mtow_kg / D.wing_area_m2
GA = GU.gust_alleviation(WS, V_C, chord=chord, aspect_ratio=REQ.aspect_ratio, sweep_deg=REQ.sweep_deg, U=10.0, rho=rho_h)
T_static = D.engines * F.scaled(D.engine_name, D.engine_scale, 0.0)[1].full(0.0)[0]
L = al.Albatros.layout(P)
tail_area = 2 * P["vtail_span"] * P["vtail_chord"] * 1e-6 * math.cos(math.radians(P["vtail_dihedral_deg"]))
VA = GU.vectoring_authority(T_static, (L["x_exit"] - L["x_ac"]) / 1000, tail_area, (L["x_tail_le"] - L["x_ac"]) / 1000)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.3))
ax[0].plot(ws, dn0, "0.6", label="unswept"); ax[0].plot(ws, dn, "C0", label=f"{REQ.sweep_deg:g}° sweep")
ax[0].axvline(WS, color="C3", ls=":"); ax[0].set_xlabel("wing loading [kg/m²]"); ax[0].set_ylabel("load factor step in a 10 m/s gust [g]", color="C0")
a2 = ax[0].twinx(); a2.plot(ws, stroke, "C1--"); a2.set_ylabel("catapult stroke at 8 g [m]", color="C1")
ax[0].legend(); ax[0].set_title("Wing loading: gusts against the catapult")
ax[1].plot(GA["t"], GA["passive"]["dn"], "0.5", label="wing alone"); ax[1].plot(GA["t"], GA["active"]["dn"], "C0", label="flaperons working")
a3 = ax[1].twinx(); a3.plot(GA["t"], GA["active"]["flap_deg"], "C2:", lw=1); a3.set_ylabel("flaperon [deg]", color="C2")
ax[1].set_xlabel("time [s]"); ax[1].set_ylabel("load factor step [g]"); ax[1].legend(); ax[1].set_title("A 10 m/s gust at the cruise")
ax[2].plot(VA["V"], VA["tail_nm"], "0.4", label="V-tail, full deflection"); ax[2].plot(VA["V"], VA["nozzles_nm"], "C3", label="nozzles 15°, full thrust")
ax[2].axvline(VA["crossover_speed"], color="C3", ls=":"); ax[2].set_ylim(0, 2.5 * VA["nozzles_nm"][0])
ax[2].set_xlabel("airspeed [m/s]"); ax[2].set_ylabel("pitching moment [N m]"); ax[2].legend(); ax[2].set_title("Thrust vectoring against the tail")
plt.tight_layout(); plt.show()
print(f"wing loading {WS:.0f} kg/m², lift slope {GU.lift_slope(REQ.aspect_ratio, REQ.sweep_deg):.2f} /rad; 10 m/s gust at {V_C:.0f} m/s: "
      f"{GA['dn_passive']:.2f} g on the wing alone, {GA['dn_active']:.2f} g with the flaperons ({GA['alleviation']:.0%} taken out, "
      f"{GA['flap_peak_deg']:.1f}° of flaperon)")
v_slow = 1.2 * D.airframe.stall_speed(D.mtow_kg)
print(f"nozzles at 15° and full thrust: {VA['nozzles_nm'][0]:.0f} N m of pitching moment at any speed = "
      f"{VA['nozzles_nm'][0] / np.interp(v_slow, VA['V'], VA['tail_nm']):.0%} of the tail's at {v_slow:.0f} m/s (the catapult's end), "
      f"{VA['nozzles_nm'][0] / np.interp(V_C * math.sqrt(rho_h / 1.225), VA['V'], VA['tail_nm']):.0%} at the cruise; equal to it at {VA['crossover_speed']:.0f} m/s")

## 7. Launch, the pass over the vessel, landing

`albatros_flight.launch`: the catapult's end speed is 1.2 × the stall speed at the take-off mass with no wind over
the deck; a ship that steams into the wind shortens the stroke. From land, the same speed on its own thrust is a
ground roll. It returns light — the cargo gone, the fuel burned — and lands slower than it left.

The release is in a pass at 1.3 × the stall speed, not from a hover (another drone does the hovering work at the
vessel). Section 8 follows the multirotor from the bay to its own flight.

In [ ]:
LA = F.launch(D) if D.concept.kind == "catapult" else None
if LA:
    display(pd.Series({"take-off mass [kg]": round(D.mtow_kg, 1), "stall speed at take-off [m/s]": round(LA["stall_speed"], 1),
                       "catapult end speed [m/s]": round(LA["launch_speed"], 1), "catapult energy [kJ]": round(LA["catapult_energy_kj"]),
                       "catapult stroke at 8 g, no wind [m]": round(LA["catapult_stroke_m"], 1),
                       "the same with 10 m/s over the deck [m]": round(LA["stroke_with_10ms_wind_m"], 1),
                       "runway take-off roll, own thrust [m]": round(LA["runway_roll_m"]),
                       "release pass over the vessel [m/s]": round(LA["release_speed"], 1),
                       "landing mass [kg]": round(LA["landing_mass_kg"], 1), "approach speed [m/s]": round(LA["approach_speed"], 1),
                       "stop at 0.35 g (skids) [m]": round(LA["landing_roll_m"])}, name="launch and landing").to_frame())
else:
    print("the chosen layout is thrust-borne: its deck numbers are in the scorecard of section 4")

## 8. The release: out of the bay, under a drogue, arms out, rotors

`albatros_drop` follows the folded hexacopter from the bay to its own hover:

1. **Ejection.** An ejector pushes the package down out of the bay. Inside the bay it is out of the airstream; as it
   comes out below the belly the drag takes hold (1.7 g aft at this speed) and its rear edge must not reach the bay's
   rear sill. That sets the ejector speed. The aircraft, lighter by the package, rises away.
2. **Free fall** for half a second, until a 3 m static line tied to the bay is taut and pulls a small **drogue** out.
3. **Under the drogue** the package hangs hub-up, slows from the pass speed and its path steepens.
4. **Unfolding.** Below 22 m/s and within 30° of the vertical the arm latch lets go. Each arm swings out in the
   hub's plane on a sprung, damped hinge and locks.
5. **Rotor start** (half a second to full thrust); the drogue is cut when the rotors carry 60 % of the weight.
6. **Recovery** to a hover, then the flight to the deck.

**Why a drogue.** The arms cannot open into the airstream of the pass: at 39 m/s the air's moment on an arm is
several times what a hinge spring of sensible size gives, and a spring strong enough would slam the lock. The drogue
takes the speed off first, and puts the hub the right way up for the rotors.

In [ ]:
PKG = DR.package(HEX)
V_REL = LA["release_speed"] if LA else 1.3 * D.airframe.stall_speed(D.mtow_kg)
M_REL = D.mtow_kg - 0.45 * FL.fuel_kg                      # the aircraft at the vessel
DROGUE, SEQ = DR.Drogue(), DR.Sequence()
EJECT = DR.eject_speed_needed(PKG, P, V_REL, M_REL)
SEP = DR.separation(PKG, P, V_REL, M_REL, eject_speed=SEQ.eject_speed)
REL = DR.simulate(PKG, V_REL, drogue=DROGUE, seq=SEQ)
ARM_CALM, ARM_PASS = DR.unfold_arm(PKG, SEQ, side_speed=0.0), DR.unfold_arm(PKG, SEQ, side_speed=V_REL)
NODROGUE = DR.without_drogue(PKG, V_REL, seq=SEQ)
display(pd.Series({
    "package: mass [kg]": round(PKG.mass_kg, 1), "package: folded L x W x H [m]": f"{PKG.length_m:.2f} x {PKG.width_m:.2f} x {PKG.height_m:.2f}",
    "pass speed at the release [m/s]": round(V_REL, 1), "ejector speed used / least for the sill [m/s]": f"{SEQ.eject_speed:.1f} / {EJECT:.1f}",
    "gap left at the bay's rear sill [mm]": round(SEP["sill_gap_m"] * 1000), "package out of the bay after [s]": round(SEP["time_out_of_bay_s"], 2),
    "least gap to the tail's belly line [m]": round(SEP["tail_gap_m"], 2), "below the aircraft when passing the tail [m]": round(SEP["drop_at_tail_m"], 1),
    "aircraft's load factor step at the release [g]": round(SEP["aircraft_dn"], 2),
    "drogue: Cd·A [m²] / canopy Ø [m]": f"{DROGUE.drag_area_m2:.2f} / {math.sqrt(4 * DROGUE.drag_area_m2 / 0.65 / math.pi):.2f}",
    "drogue out after [s]": round(REL["events"]["drogue out"], 2), "opening load [g] (limit 8)": round(REL["peak_g_opening"], 1),
    "arms released after [s], at [m/s]": f"{REL['events']['arms released']:.1f}, {REL['speed_at_unfold']:.1f}",
    "arm swing time [s] / energy into the lock [J]": f"{REL['arm']['time_s']:.2f} / {REL['arm']['lock_energy_j']:.1f}",
    "drogue cut after [s], descending at [m/s]": f"{REL['events']['drogue cut']:.1f}, {REL['descent_at_cut']:.1f}",
    "hover after [s]": round(REL["events"]["hover"], 1), "load in the recovery [g]": round(REL["peak_g_recovery"], 1),
    "height used [m]": round(REL["height_lost_m"]), "ground covered from the release point [m]": round(REL["downrange_m"]),
    "side wind an arm can open against [m/s]": round(DR.unfold_speed_limit(PKG, SEQ), 1),
    "air's moment on an arm at the pass speed / hinge spring [N m]": f"{ARM_PASS['air_moment_nm']:.1f} / {SEQ.spring_start_nm:.1f}",
    "without a drogue": "flies" if NODROGUE["ok"] else NODROGUE["events"].get("failed")}, name="the release, still air").to_frame())

tr = REL["track"]
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
ax[0].plot(tr["x"], -tr["h"], "k")
for name, t_ev in REL["events"].items():
    i = min(int(np.searchsorted(tr["t"], t_ev)), len(tr["t"]) - 1)
    ax[0].plot(tr["x"][i], -tr["h"][i], "o", ms=5); ax[0].annotate(name, (tr["x"][i], -tr["h"][i]), textcoords="offset points", xytext=(6, 2), fontsize=8)
ax[0].set_xlabel("ground distance from the release point [m]"); ax[0].set_ylabel("height below the pass [m]"); ax[0].set_aspect("equal")
ax[0].set_title("The path of the multirotor")
ax[1].plot(tr["t"], tr["speed"], "C0", label="airspeed [m/s]"); ax[1].plot(tr["t"], tr["path_deg"] / 3, "C2:", label="path from the vertical [deg / 3]")
ax[1].plot(tr["t"], tr["thrust"] / PKG.weight * 10, "C3", label="rotor thrust / weight x 10"); ax[1].plot(tr["t"], tr["g"] * 10, "0.5", lw=1, label="load [g] x 10")
ax[1].set_xlabel("time from the release [s]"); ax[1].legend(fontsize=8); ax[1].set_title("Speed, path, load, thrust")
k = ~np.isnan(SEP["gap"])
ax[2].plot(SEP["x"], -SEP["z"], "k", label="package (aircraft's frame)")
xs_ = np.linspace(0, SEP["tail_x_m"], 60)
belly = [(al.Albatros.section_scale(P, P["bay_x"] + q * 1000)[1] - al.Albatros.section_scale(P, P["bay_x"] + q * 1000)[0] * P["fuselage_height"] / 2) / 1000
         + P["fuselage_height"] / 1000 - PKG.height_m for q in xs_]
ax[2].plot(xs_, belly, "C1", lw=2, label="the aircraft's belly"); ax[2].set_xlim(-0.2, 3.5); ax[2].set_ylim(-4, 0.6)
ax[2].set_xlabel("aft of the bay's middle [m]"); ax[2].set_ylabel("below the package's start [m]"); ax[2].legend(fontsize=8)
ax[2].set_title("The first second, seen from the aircraft")
plt.tight_layout(); fig.savefig(DATA / "figures" / "50_albatros_release.png", dpi=110); plt.show()

**How big a drogue, and how high a pass.** A bigger canopy opens harder but uses less height. Then the release is
flown 400 times with what is not known scattered: the pass speed (±2 m/s), how the package tumbles before the
drogue, the canopy's size (±15 %), its delay and opening factor, the wind along the pass (5 m/s) and up or down
(2 m/s), the rotors' spin-up and thrust, and one release in fifty with a rotor that does not start (the hexacopter
then flies on four). The height that covers 99 % of them, with a quarter more and 30 m for the hover, is the height
of the pass.

In [ ]:
display(DR.drogue_trade(PKG, V_REL, seq=SEQ))
MC = DR.monte_carlo(PKG, V_REL, 400, drogue=DROGUE, seq=SEQ)
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
ax[0].hist(MC["height_lost_m"][MC["ok"]], bins=30, color="C0"); ax[0].axvline(MC["height_p99_m"], color="C3", ls=":")
ax[0].set_xlabel("height used to the hover [m]"); ax[0].set_title("Height")
ax[1].hist(MC["peak_g_opening"], bins=30, color="0.5"); ax[1].axvline(SEQ.load_limit_g, color="C3", ls=":"); ax[1].set_xlabel("opening load [g]"); ax[1].set_title("Drogue opening")
ax[2].hist(MC["downrange_m"][MC["ok"]], bins=30, color="C2"); ax[2].set_xlabel("ground covered from the release point [m]"); ax[2].set_title("Where it ends up")
plt.tight_layout(); plt.show()
print(f"{MC['success']:.1%} of {len(MC['ok'])} releases end in a hover within the load limit; failures: {MC['failures'] or 'none'}")
print(f"height used: median {MC['height_p50_m']:.0f} m, 99 % within {MC['height_p99_m']:.0f} m  ->  fly the pass at {MC['pass_height_m']:.0f} m or more above the sea")
print(f"opening load: 99 % within {MC['g_p99']:.1f} g; the hover is {MC['downrange_p05_m']:.0f}-{MC['downrange_p95_m']:.0f} m past the release point "
      f"(release that far upwind of where the multirotor should start its flight to the deck)")

The releases that do not recover are those where a rotor fails to start *and* the others give a tenth less
thrust than rated: four rotors at 1.2 times the weight, descending through their own wake. With full thrust a
hexacopter on four rotors recovers, using about twice the height. A cold pack or a heavy cargo closes that margin.

**What this leaves open.** The model is a point mass whose attitude the drogue is taken to settle; it has no
tumbling dynamics, no line snatch, no riser swing, and no bay flow in the separation. The numbers size the ejector,
the drogue, the hinge springs and the pass height for a first build; drop tests of a dummy package from the bay
replace them. The two things to watch in those tests: the package's pitch as it meets the airstream at the bay's
sill, and the drogue's line passing under the tail.

## 9. ALBATROS in CAD (Dedalus)

`designs/albatros.py`: a rounded-box fuselage around the bay (the bay's middle on the wing's aerodynamic centre, so
the trim does not jump when the multirotor leaves; a fuel tank ahead of it and one behind), a swept high wing with a
little anhedral, two engine pods on the aft shoulders — intakes above the wing's wake — and a V-tail clear of the
jets. `part="aircraft_open"` cuts the bay out; `part="multirotor"` is the bay's vehicle, unfolded.

In [ ]:
design_file = ROOT / "albatros.py"
shutil.copy(Path("../designs/albatros.py"), design_file)
shutil.copy(Path("../designs/fixed_wing.py"), ROOT / "fixed_wing.py")
albatros = dedalus.load_design(f"{design_file}:Albatros")
GEOM = albatros.generate(**dict(P, part="aircraft_open"))
MULTI = albatros.generate(**dict(P, part="multirotor"))
print(f"aircraft: valid {GEOM.shape.isValid()}, {len(GEOM.shape.Solids())} solid, {np.round(GEOM.dimensions, 0)} mm; "
      f"multirotor unfolded: {np.round(MULTI.dimensions, 0)} mm")
display(pd.Series({k: (round(v, 1) if isinstance(v, float) else v) for k, v in P.items()}, name="parameter").to_frame())

O = al.outline(P)
fig, ax = plt.subplots(2, 1, figsize=(12, 8.5), gridspec_kw={"height_ratios": [2.4, 1]})
for poly in O["plan"]:
    ax[0].fill(poly[:, 0], poly[:, 1], facecolor="0.88", edgecolor="k", lw=1)
ax[0].fill(O["bay_plan"][:, 0], O["bay_plan"][:, 1], facecolor="C1", alpha=0.5, edgecolor="C1", label="bay")
for x, y in O["rotors"]:
    ax[0].add_patch(plt.Circle((x, y), O["rotor_radius"], fill=False, ls="--", color="C0"))
ax[0].plot([O["x_ac"]], [0], "k+", ms=12, label="aerodynamic centre"); ax[0].set_aspect("equal"); ax[0].legend(); ax[0].set_title("ALBATROS, top and side [m]")
for poly in O["side"]:
    ax[1].fill(poly[:, 0], poly[:, 1], facecolor="0.88", edgecolor="k", lw=1)
ax[1].fill(O["bay_side"][:, 0], O["bay_side"][:, 1], facecolor="C1", alpha=0.5, edgecolor="C1"); ax[1].set_aspect("equal")
plt.tight_layout(); fig.savefig(DATA / "figures" / "50_albatros_views.png", dpi=110); plt.show()
try:
    dviz.show(dviz.plot3d(GEOM))
except ImportError as e:
    print("3D view needs the viz extra:", e)

In [ ]:
b = al.drag_buildup(P, 110.0, mach=110.0 / 328.6)
display(pd.DataFrame({"Cd·A [cm²]": {k: v * 1e4 for k, v in b["parts_m2"].items()}}).round(1))
m_mid = D.mtow_kg - 0.4 * FL.fuel_kg
V = np.linspace(1.15 * D.airframe.stall_speed(m_mid, rho_h), 180, 60)
_, U0, UH = F.scaled(D.engine_name, D.engine_scale, REQ.cruise_altitude_m)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(V, [D.airframe.drag(v, m_mid, rho_h) for v in V], "k", label="drag")
ax[0].plot(V, [D.engines * UH.full(v)[0] for v in V], "C3", label=f"{D.engines} engines, full thrust")
ax[0].plot(V, [UH.full(v)[0] for v in V], "C3--", label="one engine")
ax[0].set_xlabel(f"true airspeed at {REQ.cruise_altitude_m:.0f} m [m/s]"); ax[0].set_ylabel("force [N]"); ax[0].legend()
ax[1].plot(V, [m_mid * G / D.airframe.drag(v, m_mid, rho_h) for v in V], "k"); ax[1].set_ylabel("lift / drag")
a2 = ax[1].twinx()
a2.plot(V, [v / (D.engines * UH.for_thrust(v, D.airframe.drag(v, m_mid, rho_h) / D.engines)[0]) / 1000
            if UH.for_thrust(v, D.airframe.drag(v, m_mid, rho_h) / D.engines)[2] else np.nan for v in V], "C1")
a2.set_ylabel("km per kg of fuel", color="C1"); ax[1].set_xlabel("true airspeed [m/s]")
plt.tight_layout(); plt.show()

## 10. If it ends up in the water

A sea alighting is not a planned recovery for the wing-borne layout — it touches down at 30 m/s — but the hull is
sealed so that a ditching beside the ship in calm water is a recovery and not a loss: it floats high and upright,
the intakes clear of the water, to be lifted aboard by its launch hooks. In any swell the low intakes and the
wing tips decide; `wave_limit_m` is half the smallest freeboard.

In [ ]:
WATER = {"at the landing mass": GU.afloat(D, FL.landing_mass_kg), "at the take-off mass": GU.afloat(D, D.mtow_kg)}
display(pd.DataFrame({k: {q: (round(float(v), 3) if isinstance(v, (int, float, np.floating)) and not isinstance(v, bool) else v) for q, v in w.items()}
                      for k, w in WATER.items()}))

## 11. Medicine to a flooded town

A river town on the plain of eastern Poland or Lithuania stands in 2.4 m of water. People are on the roofs. ALBATROS
starts many miles away, flies its cruise, and makes **one pass** over the flood without stopping. Its bay holds six
**ÉMERILLONs** — notebook 37's electric speed quadcopter (ER2), each with a medicine pod in its tail: 0.5 kg of
vaccine in a cold box, or of solids (tablets, dressings). An ÉMERILLON is a rigid bullet built for 380 km/h: the
39 m/s of the pass is nothing to it. It needs no drogue and nothing unfolds — it falls clear for half a second,
starts its rotors and flies.

- **Pitched roofs** (the houses): it hovers tail-down a metre above the ridge, an arm's length from the people, and
  lets go a **saddle bag** — two pouches on a strap, across the ridge. A single box does not stay on wet sheet metal
  at 35-40°; the simulation shows both.
- **Flat roofs** (the school): it lands on its fins with the medicine on board.
- After a drop it flies to the school's roof and lands there to be collected.

`albatros_flood` builds the town and flies it in **MuJoCo**: the houses, roofs and ridges are contact geometry; every
ÉMERILLON is a free rigid body driven by its thrust (limited as `emerillon_flight.drive` gives it against airspeed),
a torque that turns its axis, and the air's force in a 6 m/s wind with gusts; the bags are free bodies on a chain of
ball joints that fall, strike the metal, slide and catch on the ridge — or do not. ALBATROS is a marker flying the
pass. Limits are in the module's text: no rotor wash on the bag, the strap taken as paid out at the release, the
guidance knows where the roofs are.

In [ ]:
CAR = FLD.carrier("ER2", payload_kg=0.5)
EM_OPTION = FLD.bay_option(CAR, 6, 2)
ALB_EM = F.size("catapult", replace(REQ, cargo_kg=EM_OPTION.cargo_kg), EM_OPTION)
LA_EM = F.launch(ALB_EM)
display(pd.Series({"carrier": CAR.name, "mass without / with the medicine [kg]": f"{CAR.mass_kg:.2f} / {CAR.gross_kg:.2f}",
                   "static thrust / weight": round(CAR.thrust_max(0.0) / (CAR.gross_kg * G), 1), "hover power, loaded [W]": round(CAR.hover_power()),
                   "pack [Wh] -> hover time, loaded [min]": f"{CAR.pack_wh:.0f} -> {60 * CAR.pack_wh * CAR.usable / CAR.hover_power():.1f}",
                   "in the bay: L x W x H each [m]": " x ".join(f"{v:.2f}" for v in CAR.envelope_m),
                   "bay for six, two abreast": EM_OPTION.summary()["bay L x W x H [m]"], "bay load [kg]": round(EM_OPTION.dropped_kg, 1),
                   "Albatros for this load: take-off [kg] / fuel [kg]": f"{ALB_EM.mtow_kg:.0f} / {ALB_EM.flight.fuel_kg:.0f}",
                   "the same Albatros as for the hexacopter?": f"yes in mass ({D.mtow_kg:.0f} kg there); the bay is {EM_OPTION.bay()['length_m']:.2f} m long instead of {OPTION.bay()['length_m']:.2f} m",
                   "to the flood, 300 km [min]": round(ALB_EM.flight.time_to_vessel_s / 60), "pass speed [m/s]": round(LA_EM["release_speed"], 1)},
                  name="Émerillon in Albatros").to_frame())

COLD = FLD.cold_chain(phases=(("depot and loading", 20.0, 22.0), (f"Albatros's bay at {REQ.cruise_altitude_m:.0f} m", ALB_EM.flight.time_to_vessel_s / 60, -24.0),
                              ("Émerillon to the roof", 2.0, 18.0), ("on the roof", 300.0, 24.0)))
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(COLD["t_min"], COLD["air"], "0.7", label="air around the box"); ax.plot(COLD["t_min"], COLD["T"], "C0", lw=2, label="vials")
ax.axhspan(2, 8, color="C2", alpha=0.15, label="2-8 °C"); ax.set_xlabel("time from the depot [min]"); ax.set_ylabel("°C"); ax.legend(ncol=3, fontsize=8)
ax.set_title("The vaccine box: 0.25 kg of vials, a 0.1 kg +5 °C phase-change pack, 15 mm of foam"); plt.tight_layout(); plt.show()
print(f"in 2-8 °C all the way to the roof: {COLD['in_range_until_roof']} (lowest {COLD['min_c']:.1f} °C in the cold bay: the pack freezing keeps the vials from it); "
      f"on the roof it stays in range for {COLD['hours_on_roof_in_range']:.1f} h at 24 °C")

In [ ]:
FLOOD_SCENE = FLD.Scene(pass_speed=float(LA_EM["release_speed"]))
FLOOD = FLD.run(FLOOD_SCENE)
display(FLD.table(FLOOD).T)
FSUM = FLD.summary(FLOOD)
FBOX = FLD.run(FLD.Scene(pass_speed=float(LA_EM["release_speed"]), payload="box"))
BSUM = FLD.summary(FBOX)
display(pd.DataFrame({"saddle bags": FSUM, "single boxes": BSUM}))

fig, ax = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={"width_ratios": [1.5, 1]})
ax[0].set_facecolor("#8a968c")
for q in FLOOD_SCENE.buildings:
    c, s_ = math.cos(math.radians(q.yaw_deg)), math.sin(math.radians(q.yaw_deg))
    pts = np.array([q.world(sx * q.length / 2, sy * q.width / 2) for sx, sy in ((-1, -1), (1, -1), (1, 1), (-1, 1))])
    ax[0].fill(pts[:, 0], pts[:, 1], facecolor=q.roof_colour, edgecolor="k", lw=0.6)
    if q.people:
        ax[0].plot(q.x, q.y, "o", ms=13, mfc="none", mec="yellow", mew=2)
for x, y, h in FLOOD_SCENE.trees:
    ax[0].plot(x, y, "o", color="#2f6b2a", ms=5)
for k, tr in FLOOD["track"].items():
    ax[0].plot(tr["p"][:, 0], tr["p"][:, 1], lw=1.2, label=f"Émerillon {k}")
ax[0].plot(FLOOD["log"]["mocap"][:, 0], FLOOD["log"]["mocap"][:, 1], "w--", lw=1, label="Albatros's pass")
ax[0].set_xlim(-260, 150); ax[0].set_ylim(-90, 95); ax[0].set_aspect("equal"); ax[0].legend(fontsize=7, ncol=2, loc="lower left")
ax[0].set_title("The flooded town from above: the pass, the six flights, the roofs with people (yellow rings)"); ax[0].set_xlabel("x [m]"); ax[0].set_ylabel("y [m]")
for k, tr in FLOOD["track"].items():
    ax[1].plot(tr["t"] - FLOOD["release_times"][k], tr["p"][:, 2], lw=1.2)
ax[1].axhline(FLOOD_SCENE.flood_m, color="C0", ls=":"); ax[1].set_xlabel("time from its release [s]"); ax[1].set_ylabel("height [m]"); ax[1].set_title("Down from the pass to the roofs")
plt.tight_layout(); plt.show()
print(f"saddle bags: {FSUM['delivered']} of {FSUM['roofs']} roofs, {FSUM['people reached']} of {FSUM['people waiting']} people, the last {FSUM['last delivery [s]']:.0f} s after its release; "
      f"single boxes: {BSUM['delivered']} of {BSUM['roofs']}")
print(f"from Albatros's launch {REQ.radius_km:.0f} km away to medicine on the last roof: {ALB_EM.flight.time_to_vessel_s / 60 + FSUM['last delivery [s]'] / 60:.0f} min; "
      f"the carriers used at most {FSUM['most energy used [Wh]']:.1f} of {CAR.pack_wh * CAR.usable:.0f} Wh")

**Pictures and a movie.** `MUJOCO_GL=glfw xvfb-run -a python labs/aero/sims/albatros_flood.py` renders the run with
MuJoCo's own renderer: `sims/output/albatros_flood_saddle.mp4` and stills. Four stills from the committed run are in
`designs/data/figures/50_albatros_flood_*.png`; the cell below shows them (it renders nothing itself).

In [ ]:
from IPython.display import Image as _Img
for name in ("town", "hover", "bag", "collected"):
    pth = DATA / "figures" / f"50_albatros_flood_{name}.png"
    if pth.exists():
        display(_Img(filename=str(pth), width=820))

**What the run says.**

- The release is the easy part here: a rigid ÉMERILLON catches itself within the first second and a few tens of
  metres. The pass can be flown at 100 m.
- A saddle bag stays where it is put, astride the ridge within reach; a single box slides off most of the pitched roofs (three or four in five, with the pass speed).
  The bag's landing is about 35 g on the pouch: inside the limit assumed for vials in foam (60 g), far inside the one
  for solids.
- Every carrier ends on the school's roof with nine tenths of its pack: the flight from the pass to a roof is short.
  The pack is there for the next thing — a second roof, or a longer stand-off if ALBATROS must pass beside the town.
- The vaccine box needs its phase-change pack for the *cold* bay at altitude, not only for the warm roof: without it
  the vials go below 2 °C on the way. With it they are in range to the roof and about three hours after.

**What it leaves open.** Rotor wash on a light bag and on people a metre away; finding the people (the roofs are
given here); a wet, sloping ridge under a tail-sitter that must hover precisely in gusts — the simulation's wind is
6 m/s with 1.5 m/s gusts; and that the people must reach an arm's length along a ridge to take the bag.

## 12. Export

`designs/data/albatros_design.json`: the requirements, the bay, every layout's summary and mass budget, the choice
with its gates, the chosen design's parameters, engine, fuel by phase, gust and launch numbers, the release, and
the sensitivities.

In [ ]:
def clean(v):
    if isinstance(v, (np.floating, np.integer)): return v.item()
    if isinstance(v, np.bool_): return bool(v)
    if isinstance(v, np.ndarray): return v.tolist()
    if isinstance(v, dict): return {str(k): clean(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)): return [clean(x) for x in v]
    if isinstance(v, float) and not math.isfinite(v): return None
    return v

out = {"requirements": {k: getattr(REQ, k) for k in REQ.__dataclass_fields__},
       "bay": {"option": OPTION.name, "summary": OPTION.summary(), "bay": OPTION.bay(), "multirotor": HEX.budget,
               "alternatives": {k: o.summary() for k, o in OPTIONS.items()}},
       "layouts": {k: {"summary": F.summary(d), "budget_kg": d.budget, "notes": d.notes} for k, d in DESIGNS.items()},
       "set_aside": {"lift_cruise": F.summary(ROTORS)},
       "scorecard": SCORE.to_dict(), "choice": CHOICE, "gates": F.GATES,
       "albatros": {"layout": KEY, "params": dict(P), "engine": D.engine.to_dict(), "engines": D.engines,
                    "take_off_mass_kg": D.mtow_kg, "fuel_kg": FL.fuel_kg, "fuel_l": FL.fuel_kg / 0.8, "reserve_kg": FL.fuel_left_kg,
                    "phases": FL.phases, "cruise_speed_out": FL.cruise_speed_out, "cruise_speed_back": FL.cruise_speed_back,
                    "time_to_vessel_s": FL.time_to_vessel_s, "sortie_s": FL.total_time_s, "top_speed": F.top_speed(D, REQ),
                    "cd0": D.airframe.cd0, "wing_area_m2": D.wing_area_m2, "drag_parts_m2": b["parts_m2"], "launch": LA,
                    "gust": {"wing_loading": WS, "dn_passive": GA["dn_passive"], "dn_active": GA["dn_active"],
                             "flap_peak_deg": GA["flap_peak_deg"], "vectoring_crossover_speed": VA["crossover_speed"]},
                    "afloat": WATER},
       "release": {"package": {"mass_kg": PKG.mass_kg, "folded_m": [PKG.length_m, PKG.width_m, PKG.height_m]}, "pass_speed": V_REL,
                   "drogue": {"drag_area_m2": DROGUE.drag_area_m2, "line_m": DROGUE.line_m}, "eject_speed": SEQ.eject_speed,
                   "eject_speed_needed": EJECT, "separation": {k: v for k, v in SEP.items() if not isinstance(v, np.ndarray)},
                   "events": REL["events"], "height_lost_m": REL["height_lost_m"], "downrange_m": REL["downrange_m"],
                   "peak_g_opening": REL["peak_g_opening"], "peak_g_recovery": REL["peak_g_recovery"], "arm": REL["arm"],
                   "arm_at_pass_speed": ARM_PASS, "without_drogue": NODROGUE["events"],
                   "monte_carlo": {k: v for k, v in MC.items() if not isinstance(v, np.ndarray)}},
       "flood": {"carrier": {"name": CAR.name, "mass_kg": CAR.mass_kg, "payload_kg": CAR.payload_kg, "pack_wh": CAR.pack_wh, "hover_power_w": CAR.hover_power()},
                 "bay": EM_OPTION.summary(), "albatros": F.summary(ALB_EM), "pass_speed": LA_EM["release_speed"],
                 "cold_chain": {k: v for k, v in COLD.items() if not isinstance(v, (np.ndarray, list))},
                 "saddle": {"summary": FSUM, "deliveries": FLOOD["deliveries"]}, "box": {"summary": BSUM, "deliveries": FBOX["deliveries"]}},
       "sensitivities": SENS.reset_index().to_dict("records") if SENS is not None else None,
       "assumptions": {"mass": F.MASS, "rotor_drag": al.ROTOR_DRAG}}
(DATA / "albatros_design.json").write_text(json.dumps(clean(out), indent=1))
print("wrote", DATA / "albatros_design.json")